In [32]:
from typing import Optional, Tuple, Literal, List
import numpy as np
import scipy.sparse as sp
from scipy.spatial.distance import cdist
import time

In [2]:
from vqniche.metrics.mmd import compute_mmd_score

/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/software/cellgen/team361/am84/envs/vqniche-reproducibility/lib/python3.10/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain

In [27]:
np.random.seed(0)

shape = (1000, 10)
D = np.random.randn(shape[0], shape[1])
D_hat = np.random.randn(shape[0], shape[1])

In [37]:
start = time.time()
score = compute_mmd_score(
            D=[row for row in D],
            D_hat=[row for row in D_hat],
            # parallel=True,
        )
print(f"Score: {score}")
end = time.time()
print(f"Time: {time.time() - start}")

Score: 0.002016397782233882
Time: 11.543254375457764


In [35]:
def mmd_score(
        D: List[np.ndarray],
        D_hat: List[np.ndarray],
        method: str = 'np',
    ) -> float:
    """
    Compute the Maximum Mean Discrepancy (MMD) between two collections of distributions.

    Parameters
    ----------
    - D: List[numpy.ndarray]
        A collection of distributions.
    - D_hat: List[numpy.ndarray]
        A collection of distributions.

    Returns
    -------
    - mmd: float
        The MMD between the two collections of distributions.
    """
    assert len(D) == len(D_hat)

    if method == 'np':
        compute_total_discrepancy = pure_np_total_discrepancy
    elif method == 'sp':
        compute_total_discrepancy = scipy_total_discrepancy
    
    K_XX = compute_total_discrepancy(
        X=D,
        Y=D,
    )
    K_YY = compute_total_discrepancy(
        X=D_hat,
        Y=D_hat,
    )
    K_XY = compute_total_discrepancy(
        X=D,
        Y=D_hat,
    )
    mmd = K_XX + K_YY - 2 * K_XY
    return mmd


In [33]:
def _pad_to_width(arrs: List[np.ndarray], width: int, dtype=None) -> np.ndarray:
    """Pad 1D arrays with zeros to a common width and stack -> (n, width)."""
    n = len(arrs)
    out = np.zeros((n, width), dtype=dtype or np.result_type(*arrs))
    for i, a in enumerate(arrs):
        out[i, :a.size] = a
    return out

def pure_np_total_discrepancy(
    X: List[np.ndarray],
    Y: List[np.ndarray],
    kernel: Optional[Literal['l1_gaussian_tv']] = 'l1_gaussian_tv',
    bandwidth: float = 1.0,
    dtype=np.float32,   # float32 often speeds things up enough with no loss
) -> float:
    assert kernel == 'l1_gaussian_tv', "Only 'l1_gaussian_tv' implemented here."
    # 1) Pad once
    maxw = max(max(x.size for x in X), max(y.size for y in Y))
    Xp = _pad_to_width(X, maxw, dtype=dtype)  # (n, d)
    Yp = _pad_to_width(Y, maxw, dtype=dtype)  # (m, d)

    # 2) Pairwise L1 via broadcasting, then Gaussian TV kernel, then mean
    # distances: (n, m) = sum_k |X[i,k] - Y[j,k]|
    # Use out-of-core-friendly order: expand the smaller side if they're very imbalanced.
    if Xp.shape[0] <= Yp.shape[0]:
        D = np.abs(Xp[:, None, :] - Yp[None, :, :]).sum(axis=-1)
    else:
        D = np.abs(Yp[None, :, :] - Xp[:, None, :]).sum(axis=-1).T

    K = np.exp(-(D * D) / (2.0 * (bandwidth ** 2)))
    return float(K.mean())

In [34]:
def scipy_total_discrepancy(
    X: List[np.ndarray],
    Y: List[np.ndarray],
    kernel: Optional[Literal['l1_gaussian_tv']] = 'l1_gaussian_tv',
    bandwidth: float = 1.0,
    dtype=np.float32,
) -> float:
    assert kernel == 'l1_gaussian_tv'
    maxw = max(max(x.size for x in X), max(y.size for y in Y))
    Xp = np.zeros((len(X), maxw), dtype=dtype)
    Yp = np.zeros((len(Y), maxw), dtype=dtype)
    for i, a in enumerate(X): Xp[i, :a.size] = a
    for j, b in enumerate(Y): Yp[j, :b.size] = b

    D = cdist(Xp, Yp, metric='cityblock')          # L1 distances (n, m)
    K = np.exp(-(D * D) / (2.0 * (bandwidth ** 2)))
    return float(K.mean())

In [36]:
start = time.time()
score = mmd_score(
            D=[row for row in D],
            D_hat=[row for row in D_hat],
            method='np',
            # parallel=True,
        )
print(f"Score: {score}")
end = time.time()
print(f"Time: {time.time() - start}")

Score: 0.0019991189697066147
Time: 0.2178175449371338


In [38]:
start = time.time()
score = mmd_score(
            D=[row for row in D],
            D_hat=[row for row in D_hat],
            method='sp',
            # parallel=True,
        )
print(f"Score: {score}")
end = time.time()
print(f"Time: {time.time() - start}")

Score: 0.001999118878888629
Time: 0.07683134078979492
